In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "--require-hashes", "-r", "requirements-cellvit.txt",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-cellvit-extra.txt",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub<1.0", "hf-transfer",
])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")
print("dependency setup complete")

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
DATASET = "pathmnist"  # pathmnist | histoset | skintissue
SEED = 42  # any int; one seed per run
DATA_PATH = DATA_PATHS[DATASET]

CHECKPOINT_CANDIDATES = [
    DATA_ROOT / "CellViT-256-x40-AMP.pth",
    Path("/kaggle/input/cellvit-checkpoints/CellViT-256-x40-AMP.pth"),
]
CHECKPOINT_PATH = str(
    next((p for p in CHECKPOINT_CANDIDATES if p.is_file()), CHECKPOINT_CANDIDATES[0])
)

INPUT_MPP = 0.5  # microns per pixel of the source tiles
MODEL_MPP = 0.25  # CellViT's own resolution; patches are resized to it
MAGNIFICATION = 40  # 20 | 40

CACHE_DIR = "/kaggle/working/cellvit_features"
DINO_MODEL = "facebook/dinov2-base"
BATCH_SIZE = 2  # patches per CellViT forward; 2 fits a 16 GiB T4
DINO_CROP_BATCH_SIZE = 32  # nuclei per crop-DINOv2 forward; ignored when SKIP_CROP_DINO=True
SMOKE_SAMPLES = 8  # patches in the preflight pilot
MAX_ESTIMATED_HOURS = 10.0  # abort if the pilot projects more wall clock than this
MAX_CELLS_PER_PATCH = 16  # cap on nuclei kept per patch
OVERWRITE = False  # True | False -- False refuses to clobber an existing cache

SAVE_INSTANCE_MAPS = True  # True | False -- the compressed segmentation sidecar

SKIP_CROP_DINO = True  # True | False -- True drops the crop encoder (~60% smaller cache, ~half the runtime); makes cell_source='crop_dino' unrunnable

PARALLEL = True  # True | False

MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

assert Path(DATA_PATH).exists(), DATA_PATH
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
assert MAGNIFICATION in (20, 40)
assert not str(CACHE_DIR).startswith("/kaggle/input"), "CACHE_DIR must be writable"

In [ ]:
from huggingface_hub import snapshot_download

if "/" in DINO_MODEL and not Path(DINO_MODEL).exists():
    print(f"Downloading {DINO_MODEL} ...")
    snapshot_download(repo_id=DINO_MODEL)

In [ ]:
from utils.parallel import visible_gpu_count as _visible_gpu_count

PREFLIGHT_SHARDS = max(1, _visible_gpu_count() if PARALLEL else 1)
preflight = [
    sys.executable, "scripts/preflight_cellvit.py",
    "--dataset", DATASET, "--data_path", DATA_PATH,
    "--checkpoint", CHECKPOINT_PATH, "--cache_dir", CACHE_DIR,
    "--input_mpp", str(INPUT_MPP), "--model_mpp", str(MODEL_MPP),
    "--magnification", str(MAGNIFICATION),
    "--vit_name", DINO_MODEL, "--seed", str(SEED),
    "--smoke_samples", str(SMOKE_SAMPLES),
    "--cellvit_batch_size", str(BATCH_SIZE),
    "--dino_crop_batch_size", str(DINO_CROP_BATCH_SIZE),
    "--max_estimated_hours", str(MAX_ESTIMATED_HOURS),
    "--shards", str(PREFLIGHT_SHARDS),
]
if MAX_CELLS_PER_PATCH is not None:
    preflight += ["--max_cells_per_patch", str(MAX_CELLS_PER_PATCH)]
if SKIP_CROP_DINO:
    preflight.append("--skip_crop_dino")
if SAVE_INSTANCE_MAPS:
    preflight.append("--save_instance_maps")
subprocess.check_call(preflight)

In [ ]:
import shutil

from data.npz_mmap import export_npz_to_npy
from scripts.cellvit_shard_worker import build_shard_jobs, run_cellvit_shard
from utils import nucleus_archive_stem
from utils.parallel import run_variants_parallel, visible_gpu_count

OPTIONS = {
    "dataset": DATASET,
    "data_path": DATA_PATH,
    "checkpoint": CHECKPOINT_PATH,
    "cache_dir": CACHE_DIR,
    "input_mpp": INPUT_MPP,
    "model_mpp": MODEL_MPP,
    "magnification": MAGNIFICATION,
    "vit_name": DINO_MODEL,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "dino_crop_batch_size": DINO_CROP_BATCH_SIZE,
}
if MAX_CELLS_PER_PATCH is not None:
    OPTIONS["max_cells_per_patch"] = MAX_CELLS_PER_PATCH

if DATA_PATH.endswith(".npz"):
    export_npz_to_npy(DATA_PATH, MMAP_CACHE_DIR)
    OPTIONS["mmap_cache_dir"] = MMAP_CACHE_DIR
    free = shutil.disk_usage(MMAP_CACHE_DIR).free
    print(f"mmap export ready | free disk {free / 2**30:.1f} GiB")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

SHARDS = max(1, visible_gpu_count() if PARALLEL else 1)
print(f"GPUs visible: {visible_gpu_count()} | shards: {SHARDS}")
print("cache:", Path(CACHE_DIR) / f"{DATASET}_seed{SEED}")

In [ ]:
import time

def free_gib(path="/kaggle/working"):
    return shutil.disk_usage(path).free / 2**30

def drop_mmap_export():
    """Delete the .npy export as soon as no process still needs pixels.

    It is ~15 GiB for PathMNIST-224 and is pure scratch: only the CellViT
    forward pass reads it. Assembly reads the .build shards instead, so holding
    the export through assembly is 15 GiB of a ~20 GB quota spent on nothing --
    which is exactly how a five-hour extraction died with
    `OSError: [Errno 28] No space left on device` at the merge step.
    """
    path = OPTIONS.get("mmap_cache_dir")
    if path and Path(path).is_dir():
        shutil.rmtree(path, ignore_errors=True)
        OPTIONS.pop("mmap_cache_dir", None)
        print(f"removed mmap export | free disk {free_gib():.1f} GiB")

manifest = Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
started = time.time()
print(f"free disk before extraction: {free_gib():.1f} GiB")

if manifest.is_file() and not OVERWRITE:
    print("completed cache already exists; skipping extraction:", manifest)
    drop_mmap_export()
elif SHARDS == 1:
    run_cellvit_shard(OPTIONS, overwrite=OVERWRITE,
                      save_instance_maps=SAVE_INSTANCE_MAPS,
                      skip_crop_dino=SKIP_CROP_DINO)
else:
    if OVERWRITE and (Path(CACHE_DIR) / f"{DATASET}_seed{SEED}").is_dir():
        shutil.rmtree(Path(CACHE_DIR) / f"{DATASET}_seed{SEED}", ignore_errors=True)
        print(f"cleared previous cache for {DATASET} seed {SEED}")
    jobs = build_shard_jobs(OPTIONS, SHARDS, overwrite=False,
                            save_instance_maps=SAVE_INSTANCE_MAPS,
                            skip_crop_dino=SKIP_CROP_DINO)
    results = run_variants_parallel(jobs, run_cellvit_shard, num_workers=SHARDS)
    failed = [r["label"] for r in results if not r["ok"]]
    assert not failed, f"shards failed: {failed}"

    drop_mmap_export()

    print(f"free disk before assembly: {free_gib():.1f} GiB")
    run_cellvit_shard(OPTIONS, assemble_only=True,
                      save_instance_maps=SAVE_INSTANCE_MAPS,
                      skip_crop_dino=SKIP_CROP_DINO)

drop_mmap_export()
print(f"extraction total {time.time() - started:.0f}s | free disk {free_gib():.1f} GiB")

In [ ]:
import numpy as np

from data.identity import sample_order_fingerprint
from data.loaders import get_data_loaders, get_sample_ids
from features.cellvit.cache import load_cellvit_cache
from utils import set_seed

set_seed(SEED)
train_loader, _, _ = get_data_loaders(
    DATA_PATH, SEED, verbose=True,
    mmap_cache_dir=OPTIONS.get("mmap_cache_dir"), num_workers=0,
)
expected_ids = get_sample_ids(train_loader.dataset)

cache = load_cellvit_cache(
    str(Path(CACHE_DIR) / f"{DATASET}_seed{SEED}"),
    expected_sample_ids=expected_ids,
)
assert cache.num_patches == len(expected_ids), (cache.num_patches, len(expected_ids))
assert cache.manifest["sample_fingerprint"] == sample_order_fingerprint(expected_ids)
assert cache.manifest["dataset"] == DATASET and cache.manifest["seed"] == SEED
assert cache.offsets[0] == 0 and cache.offsets[-1] == cache.num_cells

empty = int(np.sum(np.diff(cache.offsets) == 0))
print(f"OK patches={cache.num_patches} cells={cache.num_cells} "
      f"mean={cache.num_cells / max(1, cache.num_patches):.1f}/patch")
print(f"    cellvit={cache.features('cellvit_embedding').shape}")
if cache.manifest.get("has_cell_dino_features"):
    print(f"    crop_dino={cache.features('crop_dino').shape}")
else:
    print("    crop_dino: not stored (SKIP_CROP_DINO) -> "
          "the sampler must keep cell_source='cellvit_embedding'")
assert bool(cache.manifest.get("has_cell_dino_features")) is not SKIP_CROP_DINO
print(f"    patches with no nucleus: {empty} "
      f"({100 * empty / max(1, cache.num_patches):.1f}%) -> pact missing_impute")

if SAVE_INSTANCE_MAPS:
    from features.cellvit.segmaps import has_instance_maps, read_instance_map

    cache_path = Path(CACHE_DIR) / f"{DATASET}_seed{SEED}"
    assert has_instance_maps(str(cache_path)), "instance maps requested but absent"
    assert cache.manifest.get("has_instance_maps") is True

    probe = next(
        (i for i in range(cache.num_patches)
         if cache.offsets[i + 1] > cache.offsets[i]),
        0,
    )
    instance_map = read_instance_map(str(cache_path), probe)
    labels = int(len(np.unique(instance_map)) - (1 if (instance_map == 0).any() else 0))
    expected = int(cache.offsets[probe + 1] - cache.offsets[probe])
    sidecar_mb = (cache_path / "instance_maps.bin").stat().st_size / 1e6
    print(f"    instance maps: {sidecar_mb:.1f} MB total "
          f"({sidecar_mb * 1000 / max(1, cache.num_patches):.1f} KB/patch)")
    print(f"    patch {probe}: map {instance_map.shape} holds {labels} instance(s), "
          f"features hold {expected} cell(s)")
    assert labels >= expected, (labels, expected)
assert not (Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / ".build").exists(), (
    "the .build directory survives only when assembly did not finish"
)

if OPTIONS.get("mmap_cache_dir") and Path(OPTIONS["mmap_cache_dir"]).is_dir():
    freed = sum(
        f.stat().st_size
        for f in Path(OPTIONS["mmap_cache_dir"]).rglob("*") if f.is_file()
    )
    shutil.rmtree(OPTIONS["mmap_cache_dir"], ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")

In [ ]:
import json
import shutil

SOURCE = Path(CACHE_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "CACHE_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = nucleus_archive_stem(
    DATASET, SEED, Path(CHECKPOINT_PATH).stem, DINO_MODEL, MAX_CELLS_PER_PATCH
)
ARCHIVE = WORKING / STEM

leftover_build = [p for p in SOURCE.rglob(".build") if p.is_dir()]
assert not leftover_build, (
    f"assembly did not finish: {leftover_build}. Re-run the extraction cell "
    "before archiving."
)
if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file() and ".build" not in path.parts:
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.1f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)
if OPTIONS.get("mmap_cache_dir") and Path(OPTIONS["mmap_cache_dir"]).is_dir():
    shutil.rmtree(OPTIONS["mmap_cache_dir"], ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
  3. In run_al_main.ipynb: Add Data -> your new dataset. It probes for
     '{DATASET}_seed{SEED}/manifest.json' a few levels down, so there is no
     path to edit.""")